# Open Food Facts — Exploratory Analysis

This notebook walks through the streamed OFF sample using DuckDB. It is the
interactive companion to `src/profile_data.py`, `src/generate_figures.py`,
and `src/generate_report.py`.

**Data:** Open Food Facts dump, ODbL 1.0. Streamed prefix only (default 50k
records). See `docs/assumptions_and_limitations.md`.

**Reproduce:** run the cells top to bottom after `python src/download_data.py`.

In [1]:
from pathlib import Path
import sys

# Make `src` importable from inside `notebooks/`.
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

import duckdb
import pandas as pd
from src.db import connect, register_jsonl_view, KEY_COLUMNS, NUTRITION_100G_FIELDS

RAW = REPO / 'data' / 'raw' / 'openfoodfacts_sample.jsonl'
SAMPLE = REPO / 'data' / 'sample' / 'openfoodfacts_sample.jsonl'
JSONL = RAW if RAW.exists() else SAMPLE
print('Using:', JSONL)

Using: C:\Users\seyyi\Documents\Github Projekte\Data Projekte\open-food-facts-data-quality-case-study\data\raw\openfoodfacts_sample.jsonl


In [2]:
con = connect()
register_jsonl_view(con, JSONL, 'off_raw')
n_rows = con.execute('SELECT COUNT(*) FROM off_raw').fetchone()[0]
print(f'Rows loaded: {n_rows:,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 50,000


## Missing values per key field

We treat empty strings as missing for text fields, in line with our SQL.

In [3]:
rows = []
for col in list(KEY_COLUMNS) + list(NUTRITION_100G_FIELDS):
    n = con.execute(f'''
        SELECT COUNT(*) FROM off_raw
        WHERE "{col}" IS NULL
           OR (TYPEOF("{col}") = 'VARCHAR' AND TRIM(CAST("{col}" AS VARCHAR)) = '')
    ''').fetchone()[0]
    rows.append({'column': col, 'missing': n, 'missing_pct': round(100*n/n_rows, 2)})
df_missing = pd.DataFrame(rows).sort_values('missing_pct', ascending=False)
df_missing

,column,missing,missing_pct
10,ecoscore_grade,35898,71.80
19,fiber_100g,15831,31.66
16,sugars_100g,10902,21.80
17,salt_100g,9271,18.54
18,proteins_100g,9025,18.05
15,fat_100g,8984,17.97
14,energy-kcal_100g,8971,17.94
6,categories,3265,6.53
7,categories_tags,2654,5.31
8,ingredients_text,1621,3.24


## Top 15 countries

In [4]:
con.execute('''
    WITH e AS (SELECT UNNEST(countries_tags) AS tag FROM off_raw WHERE countries_tags IS NOT NULL)
    SELECT REPLACE(tag, 'en:', '') AS country, COUNT(*) AS n
    FROM e WHERE tag IS NOT NULL AND tag <> ''
    GROUP BY country ORDER BY n DESC LIMIT 15
''').df()

,country,n
0,united-states,47051
1,france,2991
2,united-kingdom,940
3,world,860
4,germany,273
5,canada,131
6,mexico,95
7,spain,89
8,bolivia,80
9,switzerland,76


## Implausible nutrient values

Macros above 100g per 100g are physically impossible.

In [5]:
con.execute('''
    SELECT
      SUM(CASE WHEN TRY_CAST(fat_100g AS DOUBLE) > 100 THEN 1 ELSE 0 END) AS fat_gt_100,
      SUM(CASE WHEN TRY_CAST(sugars_100g AS DOUBLE) > 100 THEN 1 ELSE 0 END) AS sugars_gt_100,
      SUM(CASE WHEN TRY_CAST(salt_100g AS DOUBLE) > 100 THEN 1 ELSE 0 END) AS salt_gt_100,
      SUM(CASE WHEN TRY_CAST(proteins_100g AS DOUBLE) > 100 THEN 1 ELSE 0 END) AS proteins_gt_100,
      SUM(CASE WHEN TRY_CAST("energy-kcal_100g" AS DOUBLE) > 900 THEN 1 ELSE 0 END) AS energy_gt_900
    FROM off_raw
''').df()

,fat_gt_100,sugars_gt_100,salt_gt_100,proteins_gt_100,energy_gt_900
0,3705.0,3680.0,6.0,645.0,8370.0


## Nutri-Score grade distribution

In [6]:
con.execute('''
    SELECT COALESCE(LOWER(nutriscore_grade), 'unknown') AS grade, COUNT(*) AS n
    FROM off_raw GROUP BY grade ORDER BY n DESC
''').df()

,grade,n
0,e,15424
1,a,10138
2,d,7849
3,unknown,6829
4,c,5590
5,b,3934
6,not-applicable,236


## Where to go next

Run `python -m src.generate_figures data/raw/openfoodfacts_sample.jsonl`
and `python -m src.generate_report data/raw/openfoodfacts_sample.jsonl`
to materialize all charts and the Markdown report.